# Estimate Phases Per Turn Analysis

Risk differs from games like chess or go in that a single turn consists of multiple sequential decisions rather than a single move. Each turn is structured into a series "phases" (reinforcement, attack, fortify), but the attack phase can contain an arbitrary number of individual actions.

Because of this I think the effective depth of the decision process in Risk is better measured in terms of actions per turn, rather than turns themselves.

This has important implications for the search-based methods we're using to implement our agents (Expectiminimax and MCTS). In other words, a shallow search over turns may still require exploring a deep sequence of actions, and ony a shallow search through actions may not be effective.

The goal of this analysis is to estimate the "typical number of actions per turn", so we can better understand the effective search depth required.

We approximate this by measuring the average number of actions per turn over many simulated games using the Greedy agent. This may not give us a strict upper bound, but I think the agressive nature of the Greedy agent will provide a useful baseline to estimate how far ahead our search based agents should look.

In [ ]:
from nb_setup import risk_ai_game
import random
import statistics

# count the number of phases per turn
class PhasesPerTurnCollector(risk_ai_game.GameTelemetry):
    def __init__(self):
        self.phases_per_turn = []
        self._turn_before = 0
        self._phase_before = risk_ai_game.Phase.DEPLOY
        self._p_rein = 0
        self._p_atk = 0
        self._p_for = 0

    def on_game_start(self, game_state):
        self.phases_per_turn.clear()
        self._turn_before = game_state.turn_number
        self._p_rein = self._p_atk = self._p_for = 0

    def on_before_action(self, game_state):
        self._turn_before = game_state.turn_number
        self._phase_before = game_state.phase

    def on_action(self, game_state, action, result):
        pb = self._phase_before
        if pb == risk_ai_game.Phase.DEPLOY and game_state.phase == risk_ai_game.Phase.ATTACK:
            self._p_rein = 1
        if pb == risk_ai_game.Phase.ATTACK and isinstance(action, risk_ai_game.AttackAction):
            self._p_atk += 1
        if pb == risk_ai_game.Phase.FORTIFY and isinstance(
            action, (risk_ai_game.FortifyAction, risk_ai_game.EndPhaseAction)
        ):
            self._p_for = 1
        if game_state.turn_number > self._turn_before:
            self.phases_per_turn.append(self._p_rein + self._p_atk + self._p_for)
            self._p_rein = self._p_atk = self._p_for = 0

    def on_game_end(self, game_state, winner):
        total = self._p_rein + self._p_atk + self._p_for
        if total > 0:
            self.phases_per_turn.append(total)


# run N games and collect the number of phases per turn, then calculate the average.
NUM_GAMES = 200
rng = random.Random(42_001)
all_phases = []
for _ in range(NUM_GAMES):
    telemetry = PhasesPerTurnCollector()
    risk_ai_game.run_game(
        risk_ai_game.RiskAIGameOptions(
            agents=[
                risk_ai_game.GreedyAgent(0, name="Greedy-P0"),
                risk_ai_game.GreedyAgent(1, name="Greedy-P1"),
            ],
            max_turns=500,
            verbose=False,
            random_seed=rng.randrange(1, 10**9),
            game_telemetry=telemetry,
        )
    )
    all_phases.extend(telemetry.phases_per_turn)

mean_phases = statistics.mean(all_phases)
print(f"Number of games: {NUM_GAMES} mean phases per turn: {mean_phases}")

Number of games: 200 mean phases per turn: 7.444111776447106
